# Typhoon OCR 1.5 — PDPA บน Colab

เลือก PDF **หนึ่งไฟล์** (ไทยหรืออังกฤษ) แล้ว OCR ครบทุกหน้าเป็น `results` ในหน่วยความจำ จากนั้นดู raw text ที่ต้องการ ก่อนรันเซลล์บันทึกลง Drive

ลำดับ: **1 ติดตั้ง → 2 เลือกไฟล์/พาธบันทึก → 3 โหลดโมเดล → 4 นิยามฟังก์ชัน → 5 OCR ทั้งไฟล์ → 6 ดู raw text → 7 บันทึก JSONL**

`results` มีหนึ่งระเบียนต่อ PDF หน้า พร้อม `language`, `source_file`, `pdf_page`, `text`, เวลา และธง `possibly_truncated` ข้อความใน `<figure>` อาจเป็นคำบรรยายที่โมเดลแต่งขึ้น ควรเทียบกับ PDF ก่อนใช้ทำ RAG

**ไลบรารีที่ใช้ใน lab นี้**
- แพ็กเกจภายนอก: `transformers`, `accelerate`, `typhoon-ocr`, `pdf2image`, `pypdf` (ติดตั้งในข้อ 1); `torch`, `Pillow`, `google.colab` (มีใน Colab)
- Python standard library: `json`, `pathlib`, `time`
- โปรแกรมระบบ: Poppler สำหรับแปลง PDF เป็นภาพทีละหน้า

โมเดล: [Typhoon OCR 1.5 2B](https://huggingface.co/typhoon-ai/typhoon-ocr1.5-2b) · [แพ็กเกจ typhoon-ocr](https://github.com/scb-10x/typhoon-ocr/tree/master/packages/typhoon_ocr)

## 1. ติดตั้งเครื่องมือใน Colab

รันครั้งแรกในแต่ละ Colab runtime โมเดลจะดาวน์โหลดลงเครื่องของ Colab

In [ ]:
!apt-get -qq update && apt-get -qq install -y poppler-utils
%pip -q install -U transformers accelerate typhoon-ocr pdf2image pypdf


## 2. เลือก PDF และพาธบันทึก

เปลี่ยน `LANGUAGE` เป็น `th` หรือ `en` เพื่อเลือก PDF **หนึ่งไฟล์** จาก `PDF_FILES` และแก้ `SAVE_PATH` หากต้องการชื่อไฟล์ผลลัพธ์อื่น เซลล์นี้เพียงตรวจไฟล์และนับหน้า ยังไม่ OCR หรือบันทึกผล

In [ ]:
import json
import time
from pathlib import Path

import torch
from google.colab import drive
from pdf2image import convert_from_path
from PIL import Image
from pypdf import PdfReader

In [ ]:
drive.mount('/content/drive')

# เลือก PDF หนึ่งไฟล์: 'th' หรือ 'en'
PDF_FILES = {
    'th': Path('/content/drive/MyDrive/pdpa_raw_pdf/PDPA (TH).pdf'),
    'en': Path('/content/drive/MyDrive/pdpa_raw_pdf/PDPA (Eng).pdf'),
}

LANGUAGE = 'en'
PDF_PATH = PDF_FILES[LANGUAGE]
SAVE_PATH = Path('/content/drive/MyDrive/PDPA_RAG/Data/jsonl') / f'pdpa_{LANGUAGE}_typhoon_ocr.jsonl'

MODEL_ID = 'typhoon-ai/typhoon-ocr1.5-2b'
MAX_NEW_TOKENS = 4096

if not PDF_PATH.is_file():
    raise FileNotFoundError(f'ไม่พบ PDF: {PDF_PATH}')
if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU: เลือก Runtime → Change runtime type → GPU')
TOTAL_PAGES = len(PdfReader(str(PDF_PATH)).pages)
print(f'เลือก {LANGUAGE}: {PDF_PATH.name} | {TOTAL_PAGES} หน้า')
print('GPU:', torch.cuda.get_device_name(0))
print('จะบันทึกที่:', SAVE_PATH)

## 3. โหลด Typhoon OCR 1.5 (2B)

ใช้ FP16 เพื่อให้ทำงานกับ GPU ที่ Colab ฟรีอาจจัดให้ เช่น T4 โมเดล v1.5 ต้องใช้ prompt เฉพาะของผู้พัฒนา จึงดึงจากแพ็กเกจ `typhoon-ocr`


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from typhoon_ocr import get_prompt

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
).eval()
prompt = get_prompt('v1.5')('')
print('โหลดโมเดลแล้ว:', MODEL_ID)


## 4. ฟังก์ชัน OCR

`ocr_page()` อ่าน PDF หนึ่งหน้า ส่วน `ocr_file()` วนทุกหน้าของไฟล์ที่เลือกและคืน `results` ฟังก์ชันทั้งสอง **ไม่เขียนไฟล์**

In [ ]:
def ocr_page(pdf_path: Path, page_number: int) -> dict:
    """อ่าน PDF หนึ่งหน้าเป็นข้อความ; ยังไม่บันทึกไฟล์"""
    started = time.perf_counter()
    source_image = convert_from_path(
        str(pdf_path), first_page=page_number, last_page=page_number, dpi=150,
    )[0]
    image = source_image.convert('RGB')
    source_image.close()
    image.thumbnail((1800, 1800), Image.Resampling.LANCZOS)

    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, repetition_penalty=1.1,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    raw_text = processor.decode(
        generated, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    result = {
        'pdf_page': page_number,
        'text': raw_text,
        'elapsed_seconds': round(time.perf_counter() - started, 2),
        'possibly_truncated': len(generated) >= MAX_NEW_TOKENS,
    }
    image.close()
    del inputs, outputs, generated
    torch.cuda.empty_cache()
    return result


def ocr_file(pdf_path: Path, language: str) -> list[dict]:
    """อ่านทุกหน้าของ PDF ที่เลือก แล้วคืนผลทั้งหมดในหน่วยความจำ"""
    page_count = len(PdfReader(str(pdf_path)).pages)
    results = []
    for page_number in range(1, page_count + 1):
        record = ocr_page(pdf_path, page_number)
        record.update({
            'language': language,
            'source_file': pdf_path.name,
            'section_or_article': None,
            'model': MODEL_ID,
        })
        results.append(record)
        print(f'{page_number}/{page_count}: {len(record["text"])} ตัวอักษร, '
              f'{record["elapsed_seconds"]} วินาที, '
              f'truncated={record["possibly_truncated"]}')
    return results

## 5. OCR ทั้งไฟล์ที่เลือก

รันเซลล์นี้หนึ่งครั้ง `ocr_file()` จะอ่านทุกหน้าและคืนรายการ `results` เมื่อจบ เซลล์นี้ **ยังไม่บันทึกลง Drive** ใช้เวลากับ GPU ประมาณหลักสิบวินาทีต่อหน้า

In [ ]:
# OCR ไฟล์ที่เลือกให้ครบทุกหน้า; ผลยังอยู่ในหน่วยความจำ ไม่เขียน Drive
results = []
results = ocr_file(PDF_PATH, LANGUAGE)
print(f'อ่านครบ {len(results)}/{TOTAL_PAGES} หน้าแล้ว — เลือกหน้าดูในเซลล์ถัดไป')

## 6. ดู raw text จาก `results`

แก้ `PREVIEW_PAGE` แล้วรันเซลล์นี้เพื่อพิมพ์ข้อความดิบของหน้านั้น พร้อมสรุปหน้าที่อาจถูกตัด เซลล์นี้อ่านจาก `results` เท่านั้น ไม่ OCR หรือบันทึกซ้ำ

In [ ]:
# แสดงข้อความจาก results เท่านั้น — ไม่ OCR ซ้ำ
if len(results) != TOTAL_PAGES:
    raise RuntimeError('ยัง OCR ไม่ครบทุกหน้า: รันเซลล์ก่อนหน้าให้จบก่อน')

PREVIEW_PAGE = 1  # เปลี่ยนเลขหน้านี้เพื่อดูหน้าอื่น
record = results[PREVIEW_PAGE - 1]
print(f'{record["source_file"]} | PDF หน้า {record["pdf_page"]}/{TOTAL_PAGES}')
print(f'{len(record["text"])} ตัวอักษร | truncated={record["possibly_truncated"]}')
print('--- RAW TEXT ---')
print(record['text'])

truncated_pages = [r['pdf_page'] for r in results if r['possibly_truncated']]
print(f'\nสรุป: {len(results)} หน้า | อาจถูกตัด: {truncated_pages}')

## 7. บันทึกผลลง Drive

รันหลังตรวจ `results` แล้วเท่านั้น เซลล์นี้เขียน **JSONL หนึ่งไฟล์** ตาม `SAVE_PATH`: หนึ่งบรรทัดต่อหนึ่งหน้า พร้อม metadata และ raw text โดยไม่ OCR ซ้ำ

ก่อนเขียนจะตรวจว่าได้ครบทุกหน้าและตรงกับ PDF ที่เลือก หากไฟล์มีอยู่แล้วจะไม่เขียนทับ

In [ ]:

def save_results(records: list[dict], jsonl_path: Path) -> dict:
    # ตรวจว่าผลครบและตรงกับ PDF ที่เลือก
    pages = [r["pdf_page"] for r in records]
    if pages != list(range(1, TOTAL_PAGES + 1)):
        raise ValueError(
            f"OCR ยังไม่ครบหรือเลขหน้าไม่ต่อเนื่อง: "
            f"{len(records)}/{TOTAL_PAGES} หน้า"
        )

    if any(
        r["source_file"] != PDF_PATH.name
        or r["language"] != LANGUAGE
        for r in records
    ):
        raise ValueError("results ไม่ตรงกับ PDF/LANGUAGE ที่เลือก")

    if any(not str(r["text"]).strip() for r in records):
        raise ValueError("พบหน้าที่ข้อความว่าง กรุณาตรวจผล OCR ก่อน")

    # ใช้ชื่อเดียวกัน แยกโฟลเดอร์ตามรูปแบบ
    base_dir = jsonl_path.parent.parent
    paths = {
        "jsonl": jsonl_path,
        "txt": base_dir / "txt" / f"{jsonl_path.stem}.txt",
        "md": base_dir / "md" / f"{jsonl_path.stem}.md",
    }

    existing = [str(p) for p in paths.values() if p.exists()]
    if existing:
        raise FileExistsError(
            "มีไฟล์อยู่แล้ว กรุณาเปลี่ยนชื่อ SAVE_PATH ก่อน:\n"
            + "\n".join(existing)
        )

    # JSONL: หนึ่งบรรทัดต่อหน้า พร้อม metadata
    jsonl_content = "".join(
        json.dumps(r, ensure_ascii=False) + "\n"
        for r in records
    )

    # TXT: ข้อความ OCR พร้อมตัวคั่นหน้า
    txt_content = "\n\n".join(
        f"=== {r['source_file']} | PDF หน้า {r['pdf_page']} ===\n\n"
        + r["text"]
        for r in records
    ) + "\n"

    # Markdown: เพิ่มหัวข้อเอกสารและหัวข้อหน้า
    md_content = (
        f"# {PDF_PATH.name}\n\n"
        f"ภาษา: {LANGUAGE}\n\n"
        + "\n\n---\n\n".join(
            f"## PDF หน้า {r['pdf_page']}\n\n" + r["text"]
            for r in records
        )
        + "\n"
    )

    contents = {
        "jsonl": jsonl_content,
        "txt": txt_content,
        "md": md_content,
    }

    for kind, path in paths.items():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8") as output:
            output.write(contents[kind])
        print(f"บันทึก {kind.upper()}: {path}")

    truncated = [
        r["pdf_page"] for r in records
        if r.get("possibly_truncated", False)
    ]
    if truncated:
        print(f"กรุณาตรวจหน้าที่ข้อความอาจถูกตัด: {truncated}")

    return paths


saved_paths = save_results(results, SAVE_PATH)
print(f"บันทึกครบ 3 รูปแบบ จำนวน {len(results)} หน้า")